รันได้ทั้ง macOS และ Google Colab ด้วยโค้ดชุดเดียวกัน (เลือก device อัตโนมัติ +
ค้นหา repo root อัตโนมัติ ไม่ผูกกับโฟลเดอร์ที่รัน)

โหมด:
  baseline : ใช้ expert_generator เดิม (ผูกกับ question_type แบบ hardcode)
  router   : ใช้ RouterMLP (เฟส A) เลือกชุด expert จาก V_fuse แล้ว build_expert_prompt
  both     : รันทั้งสองบน "ตัวอย่างชุดเดียวกัน" แล้วเทียบ accuracy

ตัวอย่างการใช้งาน:
  เข้าโฟลเดอร์ก่อน (ครั้งเดียว):

  cd "/Users/jaypoom/Desktop/AI Project/MoXpert/Experiments"

  ### ทดสอบเล็ก ๆ ให้แน่ใจว่ารันครบ (โมเดลเล็ก เร็ว)
  python Qwen2VL_router.py --limit 3 --qwen-model Qwen/Qwen2-VL-2B-Instruct
  ### รันเต็ม (เช่นบน Colab GPU)
  python Qwen2VL_router.py --limit 0 --qwen-model Qwen/Qwen2-VL-7B-Instruct
"""

อธิบายวิธีพิมพ์คำสั่งแต่ละโหมด รันจากโฟลเดอร์ `Experiments/` เสมอ

**เข้าโฟลเดอร์ก่อน (ครั้งเดียว):**
```bash
cd "/Users/jaypoom/Desktop/AI Project/MoXpert/Experiments"
```

## 1) โหมด `baseline` — pipeline เดิม (ไม่มี router)
ใช้ `expert_generator` เลือก prompt ตาม question_type แบบ hardcode
```bash
/opt/miniconda3/envs/moxpert/bin/python Qwen2VL_router.py --mode baseline --limit 5 --qwen-model Qwen/Qwen2-VL-2B-Instruct
```
→ ได้ `Results_baseline.csv` + accuracy + confusion matrix (ไม่โหลด router)

## 2) โหมด `router` — ใช้ RouterMLP เลือก expert
router รับ V_fuse → เลือกชุด expert → prompt-parity/`build_expert_prompt`
```bash
/opt/miniconda3/envs/moxpert/bin/python Qwen2VL_router.py --mode router --limit 5 --qwen-model Qwen/Qwen2-VL-2B-Instruct
```
→ ได้ `Results_router.csv` + accuracy + CM + **routing-agreement %**

## 3) โหมด `both` — รันทั้งสองบนตัวอย่างชุดเดียวกัน แล้วเทียบ
```bash
/opt/miniconda3/envs/moxpert/bin/python Qwen2VL_router.py --mode both --limit 5 --qwen-model Qwen/Qwen2-VL-2B-Instruct
```
→ ได้ทั้ง 2 CSV + **ตารางเทียบ accuracy baseline vs router** + routing-agreement + CM ทั้งสองโหมด

---

## แฟล็กสำคัญ

| แฟล็ก | ความหมาย | ค่าเริ่มต้น |
|---|---|---|
| `--mode` | `baseline` / `router` / `both` | `both` |
| `--limit` | จำนวน**รูป** (1 รูป ≈ 4-5 คำถาม), `0` = ทั้ง dataset | `3` |
| `--qwen-model` | `Qwen/Qwen2-VL-2B-Instruct` (Mac) / `...7B...` (Colab) | 7B |
| `--tau` | override threshold ของ router | ค่าจาก ckpt (0.175) |
| `--router-ckpt` | path โมเดล router | `Router_Network/artifacts/router_real.pt` |
| `--outdir` | โฟลเดอร์เก็บ CSV | โฟลเดอร์สคริปต์ |
| `--analyze-csv PATH` | วิเคราะห์จาก CSV เดิม **ไม่โหลดโมเดล** | — |

**บน Mac ใส่ `--qwen-model Qwen/Qwen2-VL-2B-Instruct` เสมอ** (7B หนักเกินไป) — ประมาณ ~1.8 นาที/รูป ในโหมด both, ครึ่งเดียวถ้าโหมดเดียว

**ดูผลเก่าซ้ำโดยไม่ต้องรันโมเดล** (เร็วมาก):
```bash
/opt/miniconda3/envs/moxpert/bin/python Qwen2VL_router.py --analyze-csv Results_router.csv
```

เกร็ด: อยากรันเต็ม dataset ใช้ `--limit 0` (บน Colab GPU + `--qwen-model Qwen/Qwen2-VL-7B-Instruct`)

In [ ]:
%cd "/Users/jaypoom/Desktop/AI Project/MoXpert/Experiments"
!/opt/miniconda3/envs/moxpert/bin/python Qwen2VL_router.py --mode both --limit 5 --qwen-model Qwen/Qwen2-VL-2B-Instruct

In [2]:
#สร้างรูปจาก CSV ที่รันไว้แล้ว (เร็ว ไม่โหลดโมเดล):
%cd "/Users/jaypoom/Desktop/AI Project/MoXpert/Experiments"
!/opt/miniconda3/envs/moxpert/bin/python Qwen2VL_router.py --analyze-csv Results_router.csv

/Users/jaypoom/Desktop/AI Project/MoXpert/Experiments
[analyze-csv] อ่าน 20 แถวจาก Results_router.csv

--------------------------------------------------------------------
routing-agreement (router เลือกตรง heuristic): 20/20 = 100.0%
router route ต่างจาก baseline: 0 ตัวอย่าง  (accuracy ที่ต่างจาก baseline มาจากกลุ่มนี้เท่านั้น เพราะ prompt-parity)
--------------------------------------------------------------------

####################################################################
# CONFUSION MATRIX + ACCURACY — mode = router
####################################################################

[OVERALL / router]  n=20  accuracy=1.000
  -- by count --
  true\pred |     A     B     C     D
         A |     8     0     0     0
         B |     0     7     0     0
         C |     0     0     2     0
         D |     0     0     0     3
  -- by percentage (row-normalized) --
  true\pred |     A     B     C     D
         A | 100.0   0.0   0.0   0.0
         B |   0.0 100.0   0.0   0.0
